# BrandSphere AI — Week 9: Feedback Intelligence
**CRS AI Capstone 2025–26**

Covers:
1. Feedback data schema
2. 100-record simulated dataset
3. Sentiment classification
4. Plotly analytics dashboard
5. Module refinement strategy

In [ ]:
!pip install -q pandas numpy matplotlib seaborn Pillow 2>/dev/null
import subprocess, os, sys, io
r = subprocess.run(['git','clone','--quiet','https://github.com/iblamehemer/og','/content/BrandSphere'],capture_output=True,text=True)
os.chdir('/content/BrandSphere'); sys.path.insert(0,'/content/BrandSphere')
os.makedirs('assets/sample_exports',exist_ok=True)
print('✅ Ready')

In [ ]:
!pip install -q plotly 2>/dev/null
import pandas as pd, numpy as np, random, csv, datetime
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams.update({'figure.facecolor':'#0C0D0F','axes.facecolor':'#141518','text.color':'white','axes.labelcolor':'white','xtick.color':'white','ytick.color':'white'})
import plotly.express as px, plotly.graph_objects as go
from src.feedback_engine import load_feedback, get_summary
print('✅ Libraries loaded')

## 1. Feedback Data Schema

In [ ]:
schema = {
    'timestamp':  'ISO datetime string',
    'session_id': 'UUID hex string',
    'company':    'Brand company name',
    'industry':   'One of 12 industries',
    'module':     'Which app module was rated',
    'rating':     'Integer 1–5',
    'sentiment':  'positive / neutral / negative',
    'comment':    'Optional free text',
}
print('Feedback record schema:')
for k, v in schema.items():
    print(f'  {k:15s}: {v}')

## 2. Generate Simulated Dataset

In [ ]:
MODULES    = ['Logo & Design Studio','Creative Content Hub','Campaign Analytics','Multilingual Studio','Overall Brand Kit']
COMPANIES  = ['NovaTech','LuxeMode','GreenSeed','HealthAI','RetailBoost']
INDUSTRIES = ['Technology / Software','Fashion / Apparel','Sustainability / Green Tech','Healthcare','Retail / E-commerce']
COMMENTS   = ['Great output!','Very accurate','Loved the palette','Taglines were perfect','Needs improvement','Excellent results','Intuitive and fast','Best AI tool I have used','Somewhat generic','Could be more specific to my brand']

random.seed(42)
rows = []
for i in range(100):
    rating = random.choices([1,2,3,4,5], weights=[3,5,12,45,35])[0]
    ci     = random.randint(0,4)
    rows.append({
        'timestamp':  f'2025-{random.randint(1,12):02d}-{random.randint(1,28):02d}T{random.randint(9,21):02d}:00:00',
        'session_id': f'sess_{i:04d}',
        'company':    COMPANIES[ci],
        'industry':   INDUSTRIES[ci],
        'module':     random.choice(MODULES),
        'rating':     rating,
        'sentiment':  'positive' if rating>=4 else 'neutral' if rating==3 else 'negative',
        'comment':    random.choice(COMMENTS),
    })

df = pd.DataFrame(rows)
print(f'Simulated dataset: {len(df)} records')
print(df[['module','rating','sentiment']].head(10))

## 3. Summary Statistics

In [ ]:
print(f'Average rating:  {df["rating"].mean():.2f}/5.0')
print(f'Positive:        {(df["sentiment"]=="positive").sum()} ({(df["sentiment"]=="positive").mean()*100:.0f}%)')
print(f'Neutral:         {(df["sentiment"]=="neutral").sum()} ({(df["sentiment"]=="neutral").mean()*100:.0f}%)')
print(f'Negative:        {(df["sentiment"]=="negative").sum()} ({(df["sentiment"]=="negative").mean()*100:.0f}%)')

print(f'\nRatings by module:')
module_avg = df.groupby('module')['rating'].agg(['mean','count']).round(2)
module_avg.columns = ['Avg Rating','Count']
module_avg = module_avg.sort_values('Avg Rating', ascending=False)
print(module_avg)

## 4. Plotly Analytics Dashboard

In [ ]:
fig1 = px.bar(df.groupby('module')['rating'].mean().reset_index().rename(columns={'rating':'avg_rating'}),
              x='module', y='avg_rating',
              color='avg_rating', color_continuous_scale=['#E05A5A','#C9A84C','#3ECFB2'],
              title='Average Rating by Module', range_y=[0,5])
fig1.update_layout(paper_bgcolor='#141518', plot_bgcolor='#141518', font_color='#F0EDE8', showlegend=False)
fig1.show()

fig2 = px.pie(df['sentiment'].value_counts().reset_index(), names='sentiment', values='count',
              color='sentiment', color_discrete_map={'positive':'#3ECFB2','neutral':'#C9A84C','negative':'#E05A5A'},
              hole=0.55, title='Sentiment Distribution')
fig2.update_layout(paper_bgcolor='#141518', font_color='#F0EDE8')
fig2.show()

## 5. Refinement Strategy

In [ ]:
low_rated = module_avg[module_avg['Avg Rating'] < 3.8]
print('Modules needing attention (avg < 3.8):')
for mod, row in low_rated.iterrows():
    print(f'  ⚠️  {mod}: {row["Avg Rating"]:.2f}/5.0')
print()
print('Refinement actions:')
print('  • Modules rated < 3.5 → update Gemini prompts with more specific instructions')
print('  • Collect preferred_alternative text → use as few-shot examples in next prompt iteration')
print('  • Weekly feedback aggregation → track trends in satisfaction scores over time')
print('  • High negative sentiment keywords → identify in comment text using NLTK')